# Epidemic Spread Modeling with Cellular Automata
## Capstone Project Notebook

Traditional compartmental models (SIR, SEIR) assume a **well-mixed population** — every individual is equally likely to contact any other. In reality, disease spreads through **local interactions**: you infect your family members, coworkers, and neighbours, not random strangers on the other side of a city. This spatial structure profoundly affects epidemic dynamics.

**Cellular automata (CA)** provide a natural framework for modelling spatially-explicit disease spread. Each cell on a 2D grid represents an individual (or a small subpopulation), and infection can only spread between **neighbouring cells**. This simple change from global to local interactions produces qualitatively different dynamics:

- **Wave-like epidemic fronts** — infection spreads outward from the initial focus as an expanding ring, rather than the smooth exponential rise of ODE models
- **Spatial clustering** — infected individuals form connected patches rather than being uniformly distributed
- **Stochastic extinction** — small outbreaks can die out by chance before spreading, even when the ODE model predicts an epidemic
- **Geometric effects** — barriers, boundaries, and heterogeneous terrain naturally affect spread patterns
- **Percolation thresholds** — below a critical infection probability, the disease cannot sustain a connected chain of transmission across the grid

### CA-SIR vs ODE-SIR

| Feature | ODE-SIR | CA-SIR |
|---------|---------|--------|
| **Mixing** | Homogeneous (well-mixed) | Local (nearest neighbours) |
| **Space** | None | Explicit 2D grid |
| **Stochasticity** | Deterministic | Inherently stochastic |
| **Epidemic front** | Exponential rise everywhere | Expanding spatial wave |
| **Small outbreaks** | Always grow if $R_0 > 1$ | Can die out by chance |
| **Interventions** | Global parameters | Spatially targeted (quarantine zones, barriers) |

CA epidemic models are particularly useful for studying **spatial interventions** such as quarantine zones, vaccination rings, movement restrictions, and the role of population density — questions that ODE models simply cannot address.

### In this notebook you will:
1. Understand how **cellular automata** can model spatial disease spread
2. Implement a **grid-based SIR model** step by step
3. Visualise wave-like epidemic propagation
4. Explore parameter sensitivity and stochastic variability
5. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for array operations, Matplotlib for visualisation, and define a custom colour map that maps the three SIR states to intuitive colours: green for Susceptible (healthy), red for Infected (contagious), and blue for Recovered (immune). These constants will be used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

STATE_S = 0  # Susceptible
STATE_I = 1  # Infected
STATE_R = 2  # Recovered

CMAP = ListedColormap(['#2ecc71', '#e74c3c', '#3498db'])  # S=green, I=red, R=blue
print('Imports loaded.')

---
## 1 · The CA-SIR Model

Each cell on a 2D grid represents one individual with a state: **Susceptible**, **Infected**, or **Recovered**.

### Update Rules (synchronous)

For each cell at position $(r, c)$:
1. **Susceptible**: Count infected neighbours in the **Moore neighbourhood** (8 cells). Become infected with probability $1 - (1 - p_{\text{infect}})^{n_{\text{infected}}}$, where $n_{\text{infected}}$ is the number of infected neighbours.
2. **Infected**: Recover with probability $p_{\text{recover}}$ each step.
3. **Recovered**: Stay recovered (permanent immunity).

### Key Differences from ODE-SIR
- **Spatial structure**: infection only spreads locally → wave-like fronts
- **Stochastic**: different runs produce different outcomes
- **Finite population**: discrete individuals, not continuous fractions

### Step 1: Count infected neighbours

The core of any CA model is the neighbourhood function. For each cell, we need to know how many of its neighbours are in a particular state. Here we implement a function that counts the number of **infected** cells in the **Moore neighbourhood** — the 8 cells surrounding a given position (including diagonals).

Boundary handling is important: cells at the edge of the grid have fewer than 8 neighbours. We simply skip any neighbour that would fall outside the grid bounds, effectively treating the boundary as a barrier that infection cannot cross. An alternative would be periodic (wrap-around) boundaries, which model a torus-like topology.

In [ ]:
def count_infected_neighbours(grid, r, c):
    """Count infected cells in the Moore neighbourhood."""
    rows, cols = grid.shape
    count = 0
    for dr in [-1, 0, 1]:
        for dc in [-1, 0, 1]:
            if dr == 0 and dc == 0:
                continue
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                if grid[nr, nc] == STATE_I:
                    count += 1
    return count

# Test
g = np.zeros((5, 5), dtype=int)
g[2, 2] = STATE_I
g[2, 3] = STATE_I
assert count_infected_neighbours(g, 2, 1) == 1
assert count_infected_neighbours(g, 1, 2) == 2
print('Neighbour counting test passed.')

### Step 2: One CA step

This function applies the CA-SIR transition rules to every cell in the grid simultaneously. We create a copy of the grid (`new_grid`) and write updates there, so that all transitions within a single time step are based on the *same* snapshot of the current state — no cell "sees" changes made earlier in the same step.

For each cell:
- **Susceptible cells** compute their infection probability using the formula $p = 1 - (1 - p_\text{infect})^{n}$, where $n$ is the number of infected neighbours. This models independent transmission attempts: each infected neighbour has a probability $p_\text{infect}$ of transmitting the disease, and the probability of escaping *all* of them is $(1 - p_\text{infect})^n$. The complement gives the probability of catching the disease from at least one neighbour.
- **Infected cells** recover with a fixed probability $p_\text{recover}$ per step, representing the average duration of illness ($\approx 1/p_\text{recover}$ steps).

In [ ]:
def ca_sir_step(grid, p_infect, p_recover, rng):
    """Apply one step of the CA-SIR model."""
    rows, cols = grid.shape
    new_grid = grid.copy()
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == STATE_S:
                n_inf = count_infected_neighbours(grid, r, c)
                if n_inf > 0:
                    prob = 1 - (1 - p_infect) ** n_inf
                    if rng.random() < prob:
                        new_grid[r, c] = STATE_I
            elif grid[r, c] == STATE_I:
                if rng.random() < p_recover:
                    new_grid[r, c] = STATE_R
    return new_grid

print('CA-SIR step defined.')

### Step 3: Full simulation

We now wrap everything into a complete simulation function. It initialises an $80 \times 80$ grid of susceptible individuals, seeds a small cluster of infections near the centre, and iterates the CA rules for a specified number of steps.

At each step the function records:
- **Compartment counts** ($S$, $I$, $R$) for plotting epidemic curves
- **Grid snapshots** every 10 steps for spatial visualisation

The initial infections are placed in a small $5 \times 5$ patch around the grid centre, mimicking a localised outbreak (e.g., a single household or workplace). After the simulation, we verify that the total population $S + I + R$ is conserved at every step — a basic sanity check ensuring no individuals are created or lost.

In [ ]:
def simulate_ca_sir(grid_size=80, p_infect=0.3, p_recover=0.05,
                     n_initial=5, n_steps=100, seed=42):
    """Run a CA-SIR epidemic simulation."""
    rng = np.random.default_rng(seed)
    grid = np.full((grid_size, grid_size), STATE_S, dtype=int)
    
    # Seed initial infections at centre
    centre = grid_size // 2
    for _ in range(n_initial):
        r = centre + rng.integers(-2, 3)
        c = centre + rng.integers(-2, 3)
        grid[r, c] = STATE_I
    
    s_hist, i_hist, r_hist = [], [], []
    snapshots = []
    
    for step in range(n_steps):
        s_hist.append(int(np.sum(grid == STATE_S)))
        i_hist.append(int(np.sum(grid == STATE_I)))
        r_hist.append(int(np.sum(grid == STATE_R)))
        if step % 10 == 0:
            snapshots.append((step, grid.copy()))
        grid = ca_sir_step(grid, p_infect, p_recover, rng)
    
    return {
        'susceptible': np.array(s_hist),
        'infected': np.array(i_hist),
        'recovered': np.array(r_hist),
        'snapshots': snapshots,
        'grid_size': grid_size
    }

result = simulate_ca_sir()
total = result['susceptible'] + result['infected'] + result['recovered']
assert np.all(total == total[0]), 'Population not conserved!'
print(f'Simulation complete. Peak infected: {np.max(result["infected"])}')

---
## 2 · Visualising the Epidemic

The two most informative views of a CA epidemic are:

1. **Epidemic curves** (left panel) — the time series of $S$, $I$, and $R$ counts. These are directly comparable to the output of an ODE-SIR model, but here they arise from local stochastic interactions rather than global differential equations. Look for the characteristic shape: $S$ decreases monotonically, $I$ rises to a peak then falls, and $R$ accumulates as individuals recover.

2. **Spatial snapshot** (right panel) — a snapshot of the grid near the peak of infection. Unlike ODE models, the CA produces a clear **spatial structure**: an expanding ring of red (infected) cells surrounding a blue (recovered) core, with green (susceptible) cells still untouched on the periphery. This ring-like wave front is the hallmark of spatially-explicit epidemic models.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Epidemic curves
t = np.arange(len(result['susceptible']))
ax1.plot(t, result['susceptible'], color='#2ecc71', lw=2, label='S')
ax1.plot(t, result['infected'], color='#e74c3c', lw=2, label='I')
ax1.plot(t, result['recovered'], color='#3498db', lw=2, label='R')
ax1.set_xlabel('Time step'); ax1.set_ylabel('Count')
ax1.set_title('CA-SIR Epidemic Curves', fontweight='bold')
ax1.legend()

# Snapshot at peak
peak_idx = np.argmax(result['infected'])
# Find closest snapshot
best_snap = min(result['snapshots'], key=lambda x: abs(x[0] - peak_idx))
ax2.imshow(best_snap[1], cmap=CMAP, vmin=0, vmax=2, interpolation='nearest')
ax2.set_title(f'Grid at t={best_snap[0]} (near peak)', fontweight='bold')
ax2.axis('off')

plt.tight_layout(); plt.show()

### Spatial spread over time

The sequence of grid snapshots below shows how the epidemic propagates outward from the initial focus. In the early steps, only a small cluster of cells near the centre is infected. As time progresses, the infection front expands roughly as a circle (or irregular blob, due to stochasticity). Behind the front, recovered (blue) cells form an immune barrier that the infection cannot cross again.

This wave-like behaviour is fundamentally different from ODE-SIR, where there is no spatial dimension — infection simply rises and falls as smooth curves. The CA model shows that **geometry matters**: narrow corridors of susceptible cells can channel the epidemic, while gaps in the susceptible population act as natural firebreaks.

In [ ]:
n_snaps = len(result['snapshots'])
fig, axes = plt.subplots(1, min(n_snaps, 8), figsize=(3 * min(n_snaps, 8), 3))
for ax, (step, snap) in zip(axes, result['snapshots'][:8]):
    ax.imshow(snap, cmap=CMAP, vmin=0, vmax=2, interpolation='nearest')
    n_i = np.sum(snap == STATE_I)
    ax.set_title(f't={step}, I={n_i}', fontsize=9)
    ax.axis('off')
fig.suptitle('Spatial Epidemic Wave', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()
print('The infection spreads as an expanding ring — a characteristic feature of spatial models.')

---
## 3 · Parameter Exploration

The two key parameters of the CA-SIR model are the **infection probability** $p_\text{infect}$ and the **recovery probability** $p_\text{recover}$. Their ratio roughly determines whether an epidemic takes off or fizzles out — analogous to the basic reproduction number $R_0$ in ODE models.

- **Left panel — varying $p_\text{infect}$**: Higher infection probability means each contact is more likely to transmit the disease. This increases the peak number of simultaneously infected individuals and speeds up the epidemic. At very low $p_\text{infect}$, the outbreak may fail to spread beyond the initial cluster — this is the **percolation threshold** effect unique to spatial models.

- **Right panel — varying $p_\text{recover}$**: Higher recovery probability means individuals are infectious for fewer time steps (average infectious period $\approx 1/p_\text{recover}$). Faster recovery reduces the window during which each infected cell can transmit to its neighbours, dampening the epidemic. Very fast recovery can prevent the epidemic from propagating at all.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Vary infection probability
for p_inf, col in zip([0.1, 0.2, 0.3, 0.5], ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']):
    res = simulate_ca_sir(p_infect=p_inf, seed=42)
    ax1.plot(res['infected'], color=col, lw=2, label=f'$p_{{inf}}$ = {p_inf}')
ax1.set_xlabel('Step'); ax1.set_ylabel('Infected')
ax1.set_title('Vary Infection Probability', fontweight='bold')
ax1.legend(fontsize=9)

# Vary recovery probability
for p_rec, col in zip([0.02, 0.05, 0.10, 0.20], ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']):
    res = simulate_ca_sir(p_recover=p_rec, seed=42)
    ax2.plot(res['infected'], color=col, lw=2, label=f'$p_{{rec}}$ = {p_rec}')
ax2.set_xlabel('Step'); ax2.set_ylabel('Infected')
ax2.set_title('Vary Recovery Probability', fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()

---
## 4 · Stochastic Variability

A defining feature of CA models is that they are **inherently stochastic** — even with identical parameters and grid size, different random seeds produce different epidemic trajectories. This contrasts sharply with ODE-SIR, which is deterministic and always gives the same curve for the same parameters.

Below we run 20 independent simulations with different random seeds and overlay the infected-count curves. The spread of trajectories illustrates the range of possible outcomes: some runs produce large epidemics, others produce smaller ones, and occasionally an outbreak may die out early before reaching most of the grid. This variability is especially pronounced for parameters near the critical threshold where epidemics are borderline.

Understanding this stochastic variability is essential for realistic risk assessment — a single ODE prediction gives the "average" outcome, but the CA reveals the full distribution of possibilities.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
peaks = []
for seed in range(1, 21):
    res = simulate_ca_sir(seed=seed)
    ax.plot(res['infected'], color='#e74c3c', lw=0.7, alpha=0.4)
    peaks.append(np.max(res['infected']))

ax.set_xlabel('Step'); ax.set_ylabel('Infected')
ax.set_title('20 CA-SIR Runs with Different Seeds', fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Peak infected — mean: {np.mean(peaks):.0f}, std: {np.std(peaks):.0f}')

---
## 5 · Your Tasks

Implement the basic CA-SIR model (done above) and add **at least two** features:

### Task A: Heterogeneous Population
Assign different susceptibility or recovery rates to cells (e.g., age groups, immune-compromised zones).

### Task B: Quarantine & Movement Restrictions
Implement lockdown zones where infection cannot spread across boundaries. Or implement random quarantine: infected cells are isolated with some probability.

### Task C: Environmental Factors
Create high-risk zones (hospitals, schools) with higher infection probability. Add low-risk zones (rural areas) with lower density.

### Task D: Adaptive Behaviour
Cells reduce their infection susceptibility when surrounded by many infected neighbours (social distancing response).

### Discussion points
- Show spatial wave patterns and clustering effects
- Compare CA results to ODE-SIR predictions
- Analyse how initial conditions (single seed vs multiple seeds) affect spread
- Discuss advantages of spatial models over well-mixed ODE models

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your extensions below
# ============================================================
# Task A: Heterogeneous Population — TODO
# Task B: Quarantine              — TODO
# Task C: Environmental Factors   — TODO
# Task D: Adaptive Behaviour      — TODO

---
## Recommended Reading & Journal Club

**1. White, S. H. et al. (2007)** *Modeling epidemics using cellular automata.* Applied Mathematics and Computation, 186(1), 193–202. [DOI](https://doi.org/10.1016/j.amc.2006.06.126)
→ Clear introduction to CA epidemic models with SIR variants.

**2. Sirakoulis, G. Ch. et al. (2000)** *A cellular automaton model for the effects of population movement and vaccination on epidemic propagation.* Ecological Modelling, 133(3), 209–223. [DOI](https://doi.org/10.1016/S0304-3800(00)00294-5)
→ CA model with movement and vaccination — directly relevant to Tasks B and D.

**3. Fuentes, M. A. & Kuperman, M. N. (1999)** *Cellular automata and epidemiological models with spatial dependence.* Physica A, 267(3–4), 471–486. [DOI](https://doi.org/10.1016/S0378-4371(99)00027-8)
→ Analysis of spatial effects on epidemic thresholds.

**4. Wolfram, S. (2002)** *A New Kind of Science.* Wolfram Media.
→ General reference on cellular automata.